In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor
from difflib import get_close_matches
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import pypdf
import pytesseract
from pdf2image import convert_from_path, pdfinfo_from_path
from shared_config.sqlserver import FCC_MATERIAL_DATABASE, PROD_SERVER
from shared_utils.sqlserver import append_table, get_engine, read_table, replace_table

In [ ]:
# Toggles
test = False  # set to True to use test PDFs; set to False to use production PDFs
rerun_all = (
    False  # set to True to re-run all PDFs; set to False to only process new PDFs
)

In [ ]:
test = False

PROJECT_PATH = r'C:\analytics-work\production\extract-material-cert-fcc'
if test:
    PDF_PATH = r'\\ut02sa1\programs\JSF\Engineering\Materials\MaterialLotData\MaterialCertData\CertDataFiles_Test'
else:
    # PDF_PATH = r'C:\Users\M67743\OneDrive - NGC\analytics-work\production\extract-material-cert-fcc\data\AS - ASBU FCC Engineering - CertDataFiles'
    PDF_PATH = r'C:\analytics-work\production\extract-material-cert-fcc\data\AS - ASBU FCC Engineering - CertDataFiles'

OUTPUT_PATH = rf'{PROJECT_PATH}\output\fcc_material_cert_data.xlsx'

SHAREPOINT_URL = (
    'https://ngc.sharepoint.us/teams/AS-ASBUFCCEngineering/'
    'Shared%20Documents/General/FCC%20Pre%20Cure/MaterialLotData/CertDataFiles'
)

POPPLER_PATH = r'C:\Users\M67743\AppData\Local\poppler\poppler-26.02.0\Library\bin'
TESSERACT_PATH = r'C:\Users\M67743\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

if TESSERACT_PATH:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_PATH

LOW_DPI = 160  # fast scan to locate relevant pages
HIGH_DPI = 275  # quality OCR on the pages we keep
OCR_CONFIG = r'--oem 3 --psm 6'
LOT_INFO_CONFIG = r'--oem 1 --psm 6'
MAX_WORKERS_PDF = 8  # PDFs processed concurrently
MAX_WORKERS_PAGE = (
    1  # parallel page OCR within each PDF (keep low; PDF-level is more even)
)

# Vendor lot format: 5-7 alphanumeric, must contain at least one letter AND
# one digit (rules out pure-numeric order numbers like 3554628 and pure-alpha
# words like GRUMMAN). Note: tesseract may misread 0 as O on Windows.
VENDOR_LOT_PATTERN = re.compile(r'^(?=.*[A-Z])(?=.*\d)[A-Z0-9]{5,7}$')

In [ ]:
prod_engine = get_engine(PROD_SERVER, FCC_MATERIAL_DATABASE)
material_info = read_table(prod_engine, 'SELECT * FROM dbo.material_info')

In [ ]:
def file_to_sharepoint_url(file):
    """
    Build the SharePoint URL for a PDF from its local path under PDF_PATH.

    Assumes PDF_PATH is a local OneDrive shortcut to the SharePoint folder at
    SHAREPOINT_URL -- the two must point to the same folder level so the
    relative paths line up.
    """
    rel_path = Path(file).resolve().relative_to(Path(PDF_PATH).resolve())
    rel_url = quote(str(rel_path).replace('\\', '/'))
    return f'{SHAREPOINT_URL}/{rel_url}'

In [ ]:
# OCR helpers
def _ocr_page(file, page_num, dpi, config=OCR_CONFIG):
    """Render one 1-indexed page to an image and OCR it. Returns (page_num, text)."""
    imgs = convert_from_path(
        file,
        dpi=dpi,
        poppler_path=POPPLER_PATH,
        first_page=page_num,
        last_page=page_num,
    )
    if not imgs:
        return page_num, ''
    return page_num, pytesseract.image_to_string(imgs[0], config=config)


def _ocr_many(file, page_numbers, dpi, config=OCR_CONFIG):
    """OCR a set of pages in parallel. Returns {page_num: text}."""
    page_numbers = sorted(set(page_numbers))
    out = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_PAGE) as ex:
        out = {
            pn: text
            for pn, text in ex.map(
                lambda p: _ocr_page(file, p, dpi, config), page_numbers
            )
        }  # noqa: C416
    return out

In [ ]:
# Vendor lot extraction (filename + OCR cross-check)
def vendor_lot_from_filename(file):
    """Parse vendor_lot from filename. Returns None if pattern doesn't match.

    Filenames vary in structure but consistently contain an ACE-prefixed order
    number followed by the vendor lot. Examples:
        ACE0124043_GV0J71_3554628-400.pdf                          -> GV0J71
        1781728149156_ACE0112705_AH0NPM-Northrop_Grumman_...pdf    -> AH0NPM

    Strategy: split on both _ and -, find the ACE\\d+ anchor token, then
    take the first subsequent token matching VENDOR_LOT_PATTERN.
    """
    stem = Path(file).stem
    tokens = re.split(r'[^A-Za-z0-9]', stem)

    # Find the ACE order number as an anchor
    pdf_idx = next(
        (
            i
            for i, t in enumerate(tokens)
            if re.match(r'^(ACE|FCC)\d+$', t, re.IGNORECASE)
        ),
        None,
    )

    # Search after the ACE token if found, otherwise scan everything
    search_tokens = tokens[pdf_idx + 1 :] if pdf_idx is not None else tokens
    for tok in search_tokens:
        candidate = tok.strip().upper()
        if VENDOR_LOT_PATTERN.match(candidate):
            return candidate
    return None


def resolve_vendor_lot(file, ocr_text):
    """Get vendor_lot from filename (primary) and OCR (verification).

    Filename is more reliable because it doesn't go through OCR. OCR is used to
    catch cases where the filename pattern doesn't apply or is wrong.

    Returns the filename value if available, OCR value as fallback. Logs any
    mismatch so it can be reviewed.
    """
    fn_lot = vendor_lot_from_filename(file)

    m = re.search(r'Lot Name:?\s*([A-Z0-9]{5,7})', ocr_text, re.IGNORECASE)
    ocr_lot = m.group(1).strip().upper() if m else None

    if fn_lot and ocr_lot:
        if fn_lot != ocr_lot:
            # Length difference suggests something other than 0/O substitution
            if len(fn_lot) != len(ocr_lot):
                print(
                    f'    WARNING: vendor_lot length mismatch -- filename "{fn_lot}", OCR "{ocr_lot}"'
                )
            else:
                # Same length: likely the 0/O OCR error we've been seeing
                print(
                    f'    vendor_lot OCR substitution detected: OCR "{ocr_lot}" -> filename "{fn_lot}"'
                )
        return fn_lot

    if fn_lot:
        return fn_lot
    if ocr_lot:
        print(
            f'    NOTE: vendor_lot from OCR only (filename pattern did not match): "{ocr_lot}"'
        )
        return ocr_lot

    print('    WARNING: vendor_lot could not be determined from filename or OCR')
    return None

In [ ]:
# Pass 1: find which pages hold which data
# Try text layer first (instant, works on born-digital PDFs).
# Fall back to OCR scan for photocopied/scanned PDFs with no text layer.
def _classify_pages_from_text(page_texts):
    """Given {page_num: text}, return {'lot_info': [...], 'in_process': [...]}."""
    pages = {'lot_info': [], 'in_process': []}
    for pn in sorted(page_texts):
        lower = (page_texts[pn] or '').lower()
        if 'customer part number' in lower or 'picked rolls' in lower:
            pages['lot_info'].append(pn)
        if 'in process' in lower and ('rc-avg' in lower or 'sublot:' in lower):
            pages['in_process'].append(pn)
    return pages


def _find_relevant_pages_text_layer(file):
    """Use pypdf text layer to locate relevant pages. Returns None if the PDF
    has no text layer (i.e. it's a scan and needs OCR)."""
    try:
        reader = pypdf.PdfReader(file)
    except Exception as e:
        print(f'    pypdf could not open file: {e}; falling back to OCR scan.')
        return None

    texts = {}
    any_text = False
    for pn, page in enumerate(reader.pages, 1):
        text = page.extract_text() or ''
        if text.strip():
            any_text = True
        texts[pn] = text

    if not any_text:
        return None  # scan, no text layer
    return _classify_pages_from_text(texts)


def _find_relevant_pages_ocr(file, dpi_value=LOW_DPI):
    """Low-DPI OCR scan of every page. Used as fallback when text layer fails."""
    n_pages = pdfinfo_from_path(file, poppler_path=POPPLER_PATH)['Pages']
    texts = _ocr_many(file, range(1, n_pages + 1), dpi_value)
    return _classify_pages_from_text(texts)


def find_relevant_pages(file, dpi_value=LOW_DPI):
    """Locate lot-info and in-process pages. Text layer preferred, OCR fallback."""
    pages = _find_relevant_pages_text_layer(file)
    if pages is not None and pages['in_process']:
        return pages
    # No text layer, or text layer didn't contain expected keywords -> OCR scan
    return _find_relevant_pages_ocr(file, dpi_value)

In [ ]:
# Lot-level extraction (cover page)
# ---------------------------------------------------------------------------
def extract_lot_info(text, pi_indexed):
    """Extract lot-level metadata from cover page OCR text.

    `material` is matched against the known Part Info index using fuzzy matching
    to correct OCR errors like 0/O substitution.
    `vendor_lot` is handled separately in resolve_vendor_lot() because the
    filename is a more reliable source than OCR.
    """
    info: dict[str, str | None] = {
        'material': None,
        'vendor_lot': None,  # filled in by resolve_vendor_lot()
        'mfg_date': None,
        'location': None,
    }

    m = re.search(r'Customer Part Number:?\s*([A-Z0-9\-]+)', text, re.IGNORECASE)
    if m:
        part = m.group(1).strip()
        known = list(pi_indexed.index)
        if part not in known:
            matches = get_close_matches(part, known, n=1, cutoff=0.9)
            if matches:
                # print(f'    material OCR "{part}" matched to "{matches[0]}"')
                part = matches[0]
            else:
                print(
                    f'    WARNING: material "{part}" could not be matched to any known material name'
                )
        info['material'] = part

    m = re.search(
        r'Date of Manufacture:?\s*(\d{1,2}\s*-\s*[A-Za-z]{3}\s*-\s*\d{4}'
        r'|\d{1,2}\s*-\s*\d{1,2}\s*-\s*\d{4})',
        text,
    )
    if m:
        info['mfg_date'] = re.sub(r'\s*-\s*', '-', m.group(1).strip())

    m = re.search(r'([A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+)*),\s*([A-Z]{2})\s+\d{5}', text)
    if m:
        info['location'] = f'{m.group(1)}, {m.group(2)}'

    return info


def extract_picked_sublots(text):
    """Parse 'Picked Rolls / Sublots:' into a normalized set for membership tests."""
    picked = set()
    m = re.search(
        r'Picked Rolls\s*/\s*Sublots:?\s*(.+?)'
        r'(?:\n[A-Z][a-z]|\n\s*\n|APPROVED|REQUIRED|Report Number)',
        text,
        re.DOTALL | re.IGNORECASE,
    )
    if m:
        for tok in re.findall(r'\b\d{1,4}[A-Za-z]?\b', m.group(1)):
            picked.add((tok.lstrip('0') or '0').upper())
    return picked

In [ ]:
# Per-sublot extraction (in-process pages)
# ---------------------------------------------------------------------------
def _to_float(s):
    try:
        return float(s.replace(',', '.'))
    except (ValueError, AttributeError):
        return None


def extract_inprocess(text):
    """State machine over OCR lines -> one record per sublot with RC/FAW avg."""
    records = []
    cur = None

    def flush():
        if cur and cur['rc'] is not None and cur['faw'] is not None:
            records.append(cur.copy())

    for raw in text.split('\n'):
        line = re.sub(r'\s+', ' ', raw).strip()
        if not line:
            continue

        m = re.search(
            r'Sublot:?\s*(\d{1,5}[A-Za-z]?)\.?\s*(?:Segment:?\s*([A-Z][A-Z ]+?))?\s*$',
            line,
            re.IGNORECASE,
        )
        if m:
            flush()
            seg = m.group(2).strip() if m.group(2) else None
            cur = {
                'sublot': m.group(1).upper(),
                'segment': seg,
                'rc': None,
                'faw': None,
            }
            continue

        if cur is None:
            continue

        if re.search(r'\bRC[\-\s]*AVG\b', line, re.IGNORECASE):
            v = re.search(r'(\d{2}[.,]\d)', line) or re.search(r'\b(\d{2})\b', line)
            if v:
                cur['rc'] = _to_float(v.group(1))
            continue

        if re.search(r'\bFAW[\-\s]*AVG\b', line, re.IGNORECASE):
            v = re.search(r'(\d{2,3}[.,]\d)', line) or re.search(r'\b(\d{2,3})\b', line)
            if v:
                cur['faw'] = _to_float(v.group(1))
            continue

    flush()
    return records

In [ ]:
# Get processed files
def get_processed_files(engine, table='material_cert_fcc'):
    """Return a set of pdf_path values already in SQL table."""
    try:
        df = pd.read_sql(f'SELECT DISTINCT pdf_path FROM [{table}]', engine)
        return set(df['pdf_path'].dropna())
    except Exception:
        # Table doesn't exist yet on first run
        return set()

In [ ]:
# Per-file orchestration
# ---------------------------------------------------------------------------
def process_file(file, pi_indexed):
    """Two-pass OCR a single cert PDF. Returns a DataFrame of per-sublot rows."""
    print(f'\n  Processing {Path(file).name} ...')

    pages = find_relevant_pages(file, LOW_DPI)
    if not pages['in_process']:
        retry = LOW_DPI + 50
        print(f'    No in-process pages at {LOW_DPI} DPI, retrying at {retry} DPI...')
        pages = find_relevant_pages(file, retry)
        if not pages['in_process']:
            print(f'    Still none; skipping {Path(file).name}.')
            return pd.DataFrame()

    lot_text = _ocr_many(file, pages['lot_info'], HIGH_DPI, LOT_INFO_CONFIG)
    proc_text = _ocr_many(file, pages['in_process'], HIGH_DPI, OCR_CONFIG)
    page_text = {**lot_text, **proc_text}

    lot: dict[str, str | None] = {
        'material': None,
        'vendor_lot': None,
        'mfg_date': None,
        'location': None,
    }
    picked = set()

    for pn in pages['lot_info']:
        if page_text.get(pn):
            lot = extract_lot_info(page_text[pn], pi_indexed)
            lot['vendor_lot'] = resolve_vendor_lot(file, page_text[pn])
            picked = extract_picked_sublots(page_text[pn])
            break

    rows = []
    for pn in pages['in_process']:
        if page_text.get(pn):
            rows.extend(extract_inprocess(page_text[pn]))

    if not rows:
        print('    No sublot RC/FAW rows extracted.')
        return pd.DataFrame()

    material_info_row = (
        pi_indexed.loc[lot['material']] if lot['material'] in pi_indexed.index else None
    )
    if material_info_row is None:
        print(
            f'    WARNING: "{lot["material"]}" not in material_info -- CCPT will be blank.'
        )

    resin = (
        material_info_row['resin_density'] if material_info_row is not None else None
    )
    fiber = (
        material_info_row['fiber_density'] if material_info_row is not None else None
    )

    out = []
    for r in rows:
        norm = (r['sublot'].lstrip('0') or '0').upper()
        rec = {
            'material': lot['material'],
            'vendor_lot': lot['vendor_lot'],
            'mfg_date': lot['mfg_date'],
            'location': lot['location'],
            'sublot': r['sublot'],
            'segment': r['segment'],
            'rc': r['rc'],
            'faw': r['faw'],
            'ccpt': None,
            'is_picked': 1 if (picked and norm in picked) else 0,
            'pdf_path': str(Path(file).resolve()),
            'pdf_url': file_to_sharepoint_url(file),
        }
        if resin and fiber and r['rc'] is not None and r['faw'] is not None:
            rc, faw = r['rc'], r['faw']
            rec['ccpt'] = round(
                (faw / 25400) * ((1 / fiber) + ((rc / 100) / ((1 - rc / 100) * resin))),
                8,
            )
        out.append(rec)

    df = pd.DataFrame(out)
    if df['mfg_date'].notna().any():
        df['mfg_date'] = pd.to_datetime(df['mfg_date'], errors='coerce')
    print(f'    Extracted {len(df)} sublot records.')
    return df


def build_lot_summary(df, material_info):
    """Per-(material, vendor_lot) average CCPT vs CPT targets."""
    if df.empty or df['ccpt'].isna().all():
        return pd.DataFrame()

    g = (
        df.dropna(subset=['ccpt'])
        .groupby(['material', 'vendor_lot'], as_index=False)
        .agg(ccpt_lot_avg=('ccpt', 'mean'), n_sublots=('ccpt', 'size'))
    )
    g['ccpt_lot_avg'] = g['ccpt_lot_avg'].round(8)

    pi_indexed = material_info.set_index('material')
    g['cpt_target_pos'] = g['material'].map(
        lambda m: pi_indexed.loc[m, 'cpt_target_pos'] if m in pi_indexed.index else None
    )
    g['cpt_target_neg'] = g['material'].map(
        lambda m: pi_indexed.loc[m, 'cpt_target_neg'] if m in pi_indexed.index else None
    )
    g['in_range'] = g.apply(
        lambda r: (
            None
            if pd.isna(r['cpt_target_pos'])
            else bool(r['cpt_target_neg'] <= r['ccpt_lot_avg'] <= r['cpt_target_pos'])
        ),
        axis=1,
    )
    return g

In [ ]:
# Run over the material folders
# ---------------------------------------------------------------------------
def main(material_info, prod_engine):
    """
    `material_info` -- DataFrame queried from SQL `material_info` table. Must have
                   columns: "material", "resin_density", "fiber_density",
                   "cpt_target_pos", "cpt_target_neg".
    """
    root = Path(PDF_PATH)
    folders = sorted(p for p in root.iterdir() if p.is_dir())
    if not folders:
        print(f'No material folders found in {PDF_PATH}')
        return None

    pi_indexed = material_info.set_index('material')
    processed_pdfs = (
        get_processed_files(prod_engine, table='material_cert_fcc')
        if not rerun_all
        else set()
    )
    print(f'Files already in DB: {len(processed_pdfs)}')

    all_pdfs = [
        pdf
        for folder in folders
        for pdf in sorted(folder.glob('*.pdf'))
        if str(pdf.resolve()) not in processed_pdfs
    ]
    if not all_pdfs:
        print('No new PDFs for process.')
        return None

    print(
        f'Found {len(all_pdfs)} new PDF file(s) in {len(folders)} folder(s) requiring processing...'
    )

    df_all = pd.DataFrame()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_PDF) as executor:
        futures = {
            executor.submit(process_file, pdf, pi_indexed): pdf for pdf in all_pdfs
        }
        for future in futures:
            df = future.result()
            if not df.empty:
                df_all = pd.concat([df_all, df], ignore_index=True)

    print(
        f'\n{"=" * 64}\nTOTAL: {len(df_all)} records from {len(all_pdfs)} PDF(s)\n{"=" * 64}'
    )

    if not df_all.empty:
        df_all = df_all.sort_values(['material', 'pdf_path', 'sublot']).reset_index(
            drop=True
        )
        df_all['mfg_date'] = pd.to_datetime(df_all['mfg_date']).dt.normalize()
        df_all['segment'] = df_all['segment'].str.title()
        print('Writing to DB...')
    return df_all

In [ ]:
df_all = main(material_info, prod_engine)

In [ ]:
if df_all is None or df_all.empty:
    print('No new files to analyze: skipping DB write')
elif rerun_all:
    replace_table(df_all, table_name='material_cert_fcc', engine_name=prod_engine)
else:
    append_table(df_all, table_name='material_cert_fcc', engine_name=prod_engine)